# Step 2: Subcluster and visualize main cell types

Starts from the integrated AnnData produced by Step 1 (`xenium_integrated.h5ad`) and the per-celltype subclustered objects. Each main cell type is refined into subtypes (T, B, macrophage, neuroblast), and a single labeled object (`xenium_integrated_labeled.h5ad`) is saved for downstream steps.

CSV outputs written to `data/processed/`:

- `fig1d_<celltype>.csv` (UMAP coordinates per main cell type)
- `fig2a_count.csv`, `fig2a_proportion.csv` (sample-by-cell-type composition)
- `fig2a_deltas_per_patient.csv`, `fig2a_mannwhitney_stats.csv` (DX -> PT deltas and Mann-Whitney tests)
- `ext_fig3a_T_subtypes.csv` (T-cell UMAP coordinates and subtype)
- `ext_fig3c_T_subtype_prop.csv`, `ext_fig3c_stats.csv` (T-cell subtype composition and paired Wilcoxon tests)
- `suppl_table_2_xenium_panel.csv`, `suppl_table_3_ind_T_cell_subtype_prop.csv`


## Setup and imports

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from pathlib import Path
import anndata as ad
import matplotlib as mpl
import matplotlib.cm as cm
from scipy.stats import gaussian_kde, mannwhitneyu, wilcoxon
from matplotlib.patches import Polygon
from statsmodels.stats.multitest import multipletests
import os

In [ ]:
sc.settings.verbosity = 3
np.random.seed(26)

In [ ]:
# Paths
indir       = '/path/to/integrated/data/'   # available upon request
outdir_h5ad = '/path/to/integrated/data/'   # available upon request
csvdir      = '../data/processed/'
figdir      = '../figures/'

os.makedirs(csvdir, exist_ok=True)
os.makedirs(figdir, exist_ok=True)

# Figure settings
plt.rcParams['savefig.transparent'] = True
plt.rcParams['savefig.dpi'] = 600
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['font.family'] = 'Helvetica'
mpl.rcParams['font.size'] = 6
sc.settings.figdir = figdir

## Helpers

In [ ]:
def make_unique_id(ad):
    """Make obs index unique as Barcode_SampleID."""
    if 'sample' not in ad.obs.columns:
        print("Error: 'sample' column missing.")
        return
    clean_index = ad.obs.index.str.split('-').str[0]
    ad.obs.index = clean_index + '_' + ad.obs['sample'].astype(str)
    if not ad.obs.index.is_unique:
        ad.obs_names_make_unique()


sample_mapping = {
    'Patient1_0DPHBP': 'Patient1_PT',
    'Patient1_ODNNSP': 'Patient1_DX',
    'Patient4_0DNOML': 'Patient4_PT',
    'Patient4_ODNOLY': 'Patient4_DX',
    'Patient2_ODNNA3': 'Patient2_PT',
    'Patient2_ODNNBG': 'Patient2_DX',
}

def normalize_sample_meta(ad_obj):
    """Map sample codes -> Patient_X_DX/PT and add patient/timepoint cols."""
    ad_obj.obs['sample'] = ad_obj.obs['sample'].astype(str).replace(sample_mapping).astype('category')
    ad_obj.obs['patient']   = ad_obj.obs['sample'].astype(str).str.replace(r'_(DX|PT)$', '', regex=True)
    ad_obj.obs['timepoint'] = ad_obj.obs['sample'].astype(str).str.extract(r'_(DX|PT)$')[0]
    return ad_obj


# Canonical main-celltype ordering and colour palette.
order_celltype = ['Endothelial', 'Fibroblast', 'Schwann', 'Neuroblast', 'Macrophage', 'B', 'T']
color_map = {
    'Endothelial': '#d73027', 'Fibroblast': '#f46d43', 'Schwann': '#fdae61',
    'Neuroblast':  '#fee090', 'Macrophage': '#e0f3f8',
    'B': '#abd9e9', 'T': '#74add1',
}

# T-cell subtype palette: matplotlib's YlGnBu colormap sampled at
# np.linspace(0, 1, 5), assigned to subtypes in alphabetical order.
# Both 'γδT' and the ASCII spelling 'gdT' point to the same colour.
T_colors = {
    'Cytotoxic T':     '#ffffd9',
    'Naive/CM T':      '#c6e9b4',
    'Proliferating T': '#40b5c4',
    'Treg':            '#225da8',
    'γδT':             '#081d58', 'gdT': '#081d58',
}

## Load integrated + subclustered h5ad inputs

In [ ]:
adata       = sc.read_h5ad(indir + 'xenium_integrated.h5ad');       make_unique_id(adata)
adata_T     = sc.read_h5ad(indir + 'T_subclustered.h5ad');          make_unique_id(adata_T)
adata_B     = sc.read_h5ad(indir + 'B_subclustered.h5ad');          make_unique_id(adata_B)
adata_neuro = sc.read_h5ad(indir + 'Neuroblast_subclustered.h5ad'); make_unique_id(adata_neuro)
adata_macro = sc.read_h5ad(indir + 'Macrophage_subclustered.h5ad'); make_unique_id(adata_macro)

## Label T cells

In [ ]:
cluster_col = 'leiden_T_0.5'

sc.pl.violin(adata_T, ['CD3E'], groupby=cluster_col, size=0)
sc.pl.umap(adata_T, color=cluster_col, show=True)

# Remove contaminated cluster 8
adata_T = adata_T[adata_T.obs[cluster_col] != '8'].copy()

# Dotplot to confirm subtype calls
genes_to_plot = ['CD8A', 'CD4', 'GZMH', 'KLRD1', 'IL7R', 'TCF7', 'SELL', 'CCR7', 'MKI67',
                 'FOXP3', 'CTLA4', 'TIGIT', 'TRGC2']
sc.pl.dotplot(adata_T, genes_to_plot, groupby=cluster_col, standard_scale='var',
              cmap='YlOrRd', title='T cell Subtypes Marker Expression')

T_subtypes = {
    0: 'Cytotoxic T', 1: 'Naive/CM T', 2: 'γδT', 3: 'Naive/CM T', 4: 'Naive/CM T',
    5: 'Naive/CM T', 6: 'Cytotoxic T', 7: 'γδT', 9: 'Treg', 10: 'Proliferating T'
}
T_colors = {
    'Proliferating T': '#d7191c', 'Cytotoxic T': '#fdae61', 'γδT': '#ffffbf',
    'Naive/CM T': '#abd9e9', 'Treg': '#2c7bb6',
}

subtype_map = {str(k): v for k, v in T_subtypes.items()}
adata_T.obs['T_subtype'] = adata_T.obs['leiden_T_0.5'].astype(str).map(subtype_map)

# Propagate to parent adata; drop contaminated T cells (T but unassigned subtype)
adata.obs['T_subtype'] = 'NA'
adata.obs.loc[adata_T.obs.index, 'T_subtype'] = adata_T.obs['T_subtype']
mask_to_remove = (adata.obs['celltype'] == 'T') & (adata.obs['T_subtype'] == 'NA')
adata = adata[~mask_to_remove].copy()

adata_T.write_h5ad(outdir_h5ad + 'T_labeled.h5ad', compression='gzip')

## Label B cells

In [ ]:
cluster_col = 'leiden_B_0.3'

sc.pl.umap(adata_B, color=cluster_col, show=True)
genes_to_plot = ['CD19', 'MS4A1', 'CD86', 'IGHM', 'CCR7', 'CD27', 'CD38',
                 'MKI67', 'FLT3', 'IDO1', 'IL3RA', 'IRF8']
sc.pl.dotplot(adata_B, genes_to_plot, groupby=cluster_col, standard_scale='var',
              cmap='YlOrRd', title='B cell Subtypes Marker Expression')

adata_B = adata_B[adata_B.obs[cluster_col] != '5'].copy()

B_subtypes = {
    0: 'Naive B', 1: 'Naive B', 2: 'Plasma', 3: 'pDC',
    4: 'Activated B', 6: 'cDC', 7: 'Proliferating memory B'
}
B_colors = {
    'Naive B': '#d73027', 'Activated B': '#fc8d59', 'Plasma': '#fee090',
    'pDC': '#e0f3f8', 'cDC': '#91bfdb', 'Proliferating memory B': '#4575b4'
}
subtype_map = {str(k): v for k, v in B_subtypes.items()}
adata_B.obs['B_subtype'] = adata_B.obs['leiden_B_0.3'].astype(str).map(subtype_map)

adata.obs['B_subtype'] = 'NA'
adata.obs.loc[adata_B.obs.index, 'B_subtype'] = adata_B.obs['B_subtype']
mask_to_remove = (adata.obs['celltype'] == 'B') & (adata.obs['B_subtype'] == 'NA')
adata = adata[~mask_to_remove].copy()

adata_B.write_h5ad(outdir_h5ad + 'B_labeled.h5ad', compression='gzip')

## Label macrophages

In [ ]:
cluster_col = 'leiden_Macrophage_0.3'

sc.pl.umap(adata_macro, color=cluster_col, show=True)
genes_to_plot = ['C1QC', 'CD68', 'CCL4', 'F13A1', 'RNASE1', 'HS3ST2', 'CYP27A1',
                 'IL18', 'IRF8', 'TOP2A', 'MKI67', 'VCAN']
sc.pl.dotplot(adata_macro, genes_to_plot, groupby=cluster_col, standard_scale='var',
              cmap='YlOrRd', title='Macrophage Subtypes Marker Expression')

macro_subtypes = {
    0: 'Pro-inflammatory Mφ', 1: 'Proliferating Mφ', 2: 'HS3ST2 Mφ',
    3: 'Proliferating Mφ', 4: 'VCAN Mφ', 5: 'C1QC Mφ', 6: 'F13A1 Mφ',
    7: 'Pro-inflammatory Mφ', 8: 'CCL4 Mφ'
}
macro_colors = {
    'Pro-inflammatory Mφ': '#d73027', 'Proliferating Mφ': '#fc8d59',
    'VCAN Mφ': '#fee090', 'F13A1 Mφ': '#ffffbf',
    'HS3ST2 Mφ': '#e0f3f8', 'C1QC Mφ': '#91bfdb', 'CCL4 Mφ': '#4575b4',
}
subtype_map = {str(k): v for k, v in macro_subtypes.items()}
adata_macro.obs['macro_subtype'] = adata_macro.obs['leiden_Macrophage_0.3'].astype(str).map(subtype_map)

adata.obs['macro_subtype'] = 'NA'
adata.obs.loc[adata_macro.obs.index, 'macro_subtype'] = adata_macro.obs['macro_subtype']

adata_macro.write_h5ad(outdir_h5ad + 'Macrophage_labeled.h5ad', compression='gzip')

## Label neuroblasts

In [ ]:
cluster_col = 'leiden_Neuroblast_0.5'

sc.pl.umap(adata_neuro, color=cluster_col, show=True)
genes_to_plot = ['PHOX2B', 'HAND2', 'KCNQ3', 'FMN1', 'PPP2R2C', 'TH', 'DBH', 'GCH1',
                 'MKI67', 'TOP2A', 'EZH2', 'BEX1', 'NEFL', 'MYCN', 'COL1A1', 'COL4A1']
sc.pl.dotplot(adata_neuro, genes_to_plot, groupby=cluster_col, standard_scale='var',
              cmap='YlOrRd', title='Neuroblast Subtypes Marker Expression')

neuro_subtypes = {
    0: 'ADRN_Dopaminergic', 1: 'ADRN_Proliferating', 2: 'ADRN_Dopaminergic',
    3: 'ADRN_Dopaminergic', 4: 'ADRN_Calcium', 5: 'ADRN_Proliferating',
    6: 'Differentiated', 7: 'ADRN_Calcium', 8: 'MYCN', 9: 'MYCN',
}
neuro_colors = {
    'ADRN_Dopaminergic': '#d7191c', 'ADRN_Proliferating': '#fdae61',
    'ADRN_Calcium': '#ffffbf', 'Differentiated': '#abd9e9', 'MYCN': '#2c7bb6',
}
subtype_map = {str(k): v for k, v in neuro_subtypes.items()}
adata_neuro.obs['neuro_subtype'] = adata_neuro.obs['leiden_Neuroblast_0.5'].astype(str).map(subtype_map)

adata.obs['neuro_subtype'] = 'NA'
adata.obs.loc[adata_neuro.obs.index, 'neuro_subtype'] = adata_neuro.obs['neuro_subtype']

adata_neuro.write_h5ad(outdir_h5ad + 'Neuroblast_labeled.h5ad', compression='gzip')

## Combine cell-type labels and write labeled integrated AnnData

Merges the main `celltype` annotation with subtype labels and writes the
combined object (`xenium_integrated_labeled.h5ad`) used by Steps 3-7.

In [ ]:
adata.obs['celltypes_all'] = adata.obs['celltype'].astype(str)
for col in ['T_subtype', 'B_subtype', 'macro_subtype', 'neuro_subtype']:
    if col in adata.obs.columns:
        mask = (adata.obs[col] != 'NA')
        adata.obs.loc[mask, 'celltypes_all'] = adata.obs.loc[mask, col]

adata.obs['celltype'] = adata.obs['celltype'].astype('category').cat.reorder_categories(order_celltype)
adata.uns['celltype_colors'] = [color_map[cat] for cat in adata.obs['celltype'].cat.categories]

# Sample mapping (anonymised Patient IDs) and per-cell metadata.
adata = normalize_sample_meta(adata)

# Save the labelled integrated AnnData.
adata.write_h5ad(outdir_h5ad + 'xenium_integrated_labeled.h5ad', compression='gzip')

## UMAP coordinates per main cell type (Fig 1d source data)

Write `fig1d_<celltype>.csv` with columns `id, umap1, umap2, celltype` for
each main cell type. The neuroblast coordinates are split into two halves
because the source-data spreadsheet keeps them in two parts.

In [ ]:
adata = sc.read_h5ad(outdir_h5ad + 'xenium_integrated_labeled.h5ad')

umap_df = pd.DataFrame({
    'id':       adata.obs_names,
    'umap1':    adata.obsm['X_umap'][:, 0],
    'umap2':    adata.obsm['X_umap'][:, 1],
    'celltype': adata.obs['celltype'].values,
})

celltype_csv_map = {
    'Endothelial': 'fig1d_Endothelial.csv',
    'Fibroblast':  'fig1d_Fibroblast.csv',
    'Schwann':     'fig1d_Schwann.csv',
    'Macrophage':  'fig1d_Macrophage.csv',
    'B':           'fig1d_B.csv',
    'T':           'fig1d_T.csv',
}

for ct, subdf in umap_df.groupby('celltype'):
    if ct == 'Neuroblast':
        mid = len(subdf) // 2
        subdf.iloc[:mid].to_csv(csvdir + 'fig1d_Neuroblast_pt1.csv', index=False)
        subdf.iloc[mid:].to_csv(csvdir + 'fig1d_Neuroblast_pt2.csv', index=False)
    elif ct in celltype_csv_map:
        subdf.to_csv(csvdir + celltype_csv_map[ct], index=False)

In [ ]:
# UMAP coloured by main cell type, rendered from the fig1d CSVs.
fig1d_files = ['fig1d_Endothelial.csv', 'fig1d_Fibroblast.csv', 'fig1d_Schwann.csv',
               'fig1d_Neuroblast_pt1.csv', 'fig1d_Neuroblast_pt2.csv',
               'fig1d_Macrophage.csv', 'fig1d_B.csv', 'fig1d_T.csv']
umap_df = pd.concat([pd.read_csv(csvdir + f) for f in fig1d_files], ignore_index=True)

fig, ax = plt.subplots(figsize=(10/2.54, 8/2.54))
for ct, sub in umap_df.groupby('celltype'):
    ax.scatter(sub['umap1'], sub['umap2'], s=2, c=color_map.get(ct, '#999999'),
               linewidths=0, alpha=0.8, label=ct)
ax.set_xticks([]); ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)
fig.savefig(figdir + 'umap_celltype.png', dpi=600, bbox_inches='tight', transparent=True)
plt.show()

## Per-sample cell-type composition (Fig 2a source data)

Counts and proportions of each main cell type per sample, saved as
`fig2a_count.csv` and `fig2a_proportion.csv`.

In [ ]:
adata = sc.read_h5ad(outdir_h5ad + 'xenium_integrated_labeled.h5ad')

comp      = pd.crosstab(adata.obs['sample'], adata.obs['celltype'])
comp_prop = comp.div(comp.sum(axis=1), axis=0)

sample_order = [
    'Patient5_PT','Patient5_DX','Patient4_PT','Patient4_DX',
    'Patient3_PT','Patient3_DX','Patient2_PT','Patient2_DX',
    'Patient1_PT','Patient1_DX',
]
comp      = comp.reindex(sample_order)
comp_prop = comp_prop.reindex(sample_order)

# Save per-sample counts and proportions of each main cell type.
comp.reset_index().to_csv(csvdir + 'fig2a_count.csv', index=False)
comp_prop.reset_index().to_csv(csvdir + 'fig2a_proportion.csv', index=False)

In [ ]:
# Fig 2a: stacked bar with alluvial ribbons connecting DX and PT.
comp_prop = pd.read_csv(csvdir + 'fig2a_proportion.csv').set_index('sample')
comp      = pd.read_csv(csvdir + 'fig2a_count.csv').set_index('sample')

order = comp_prop.index.tolist()
meta = pd.DataFrame({'sample': order})
meta['patient']   = meta['sample'].str.replace(r'_(DX|PT)$', '', regex=True)
meta['timepoint'] = meta['sample'].str.extract(r'_(DX|PT)$')[0]
meta = meta.set_index('sample')

seg = {}
for i, sample in enumerate(order):
    cum = 0
    seg[sample] = {}
    for celltype in comp_prop.columns:
        w = comp_prop.loc[sample, celltype]
        seg[sample][celltype] = (cum, cum + w)
        cum += w

fig, ax = plt.subplots(figsize=(15.5/2.54, 8.5/2.54))
comp_prop.loc[order].plot(kind='barh', stacked=True, color=color_map, ax=ax)
bar_h = 0.5

for patient in meta['patient'].unique():
    samples = [s for s in order if (s in meta.index and meta.loc[s, 'patient'] == patient)]
    dx = [s for s in samples if meta.loc[s, 'timepoint'] == 'DX']
    pt = [s for s in samples if meta.loc[s, 'timepoint'] == 'PT']
    if len(dx) == 0 or len(pt) == 0:
        continue
    s_dx, s_pt = dx[0], pt[0]
    y_dx, y_pt = order.index(s_dx), order.index(s_pt)
    if y_dx < y_pt:
        s_upper, y_upper, s_lower, y_lower = s_dx, y_dx, s_pt, y_pt
    else:
        s_upper, y_upper, s_lower, y_lower = s_pt, y_pt, s_dx, y_dx
    y_start = y_upper + bar_h / 2.0
    y_end   = y_lower - bar_h / 2.0
    for celltype in comp_prop.columns:
        x1a, x1b = seg[s_upper][celltype]
        x2a, x2b = seg[s_lower][celltype]
        poly = Polygon([(x1a, y_start), (x1b, y_start), (x2b, y_end), (x2a, y_end)],
                       closed=True, facecolor=color_map[celltype], alpha=0.5,
                       edgecolor='none', zorder=0)
        ax.add_patch(poly)

for i, sample in enumerate(order):
    cumulative_width = 0
    for celltype in comp_prop.columns:
        width = comp_prop.loc[sample, celltype]
        count = comp.loc[sample, celltype]
        if width > 0.02:
            ax.text(cumulative_width + width / 2, i, f'{int(count)}\n({width:.2f})',
                    ha='center', va='center', fontsize=5)
        cumulative_width += width

ax.set_xlabel('Proportion'); ax.set_ylabel('Sample')
ax.set_title('Cell Type Composition per Sample')
if ax.legend_:
    ax.legend_.remove()
ax.grid(False); ax.set_frame_on(False)
for spine in ax.spines.values():
    spine.set_visible(False)
plt.tight_layout()
plt.savefig(figdir + 'celltype_composition_barplot_alluvial.pdf', format='pdf', bbox_inches='tight')
plt.show()

## Per-patient DX -> PT deltas and Mann-Whitney tests (Fig 2a)

Per-patient differences in cell-type proportion between DX and PT, together with two-sided Mann-Whitney U tests of those deltas against zero, are written to `fig2a_deltas_per_patient.csv` and `fig2a_mannwhitney_stats.csv`.

In [ ]:
comp_prop = pd.read_csv(csvdir + 'fig2a_proportion.csv').set_index('sample')

df = comp_prop.copy()
df.index = df.index.astype(str).str.strip()

meta = df.reset_index().rename(columns={'index': 'sample'})
parts = meta['sample'].str.rsplit('_', n=1, expand=True)
meta['patient']   = parts[0].str.strip()
meta['timepoint'] = parts[1].str.strip().str.upper()
meta = meta[meta['timepoint'].isin(['DX', 'PT'])].copy()

meta['neuro_prop']  = meta['Neuroblast']
meta['immune_prop'] = meta['Macrophage'] + meta['B'] + meta['T']
meta_agg = meta.groupby(['patient', 'timepoint'], as_index=False)[['neuro_prop', 'immune_prop']].mean()

neuro_wide  = meta_agg.pivot(index='patient', columns='timepoint', values='neuro_prop')
immune_wide = meta_agg.pivot(index='patient', columns='timepoint', values='immune_prop')

paired = pd.DataFrame(index=neuro_wide.index.union(immune_wide.index))
paired['delta_neuro_PT_minus_DX']  = neuro_wide['PT']  - neuro_wide['DX']
paired['delta_immune_PT_minus_DX'] = immune_wide['PT'] - immune_wide['DX']
paired = paired.dropna(subset=['delta_neuro_PT_minus_DX', 'delta_immune_PT_minus_DX']).copy()

rebuilding_patients = ['Patient1', 'Patient2', 'Patient3']
desert_patients     = ['Patient4', 'Patient5']
paired['trajectory_group'] = np.where(
    paired.index.isin(rebuilding_patients), 'immune_rebuilding',
    np.where(paired.index.isin(desert_patients), 'immune_desertification', np.nan)
)
paired = paired.dropna(subset=['trajectory_group']).copy()

rb_neuro = paired.loc[paired['trajectory_group'] == 'immune_rebuilding',     'delta_neuro_PT_minus_DX'].values
ds_neuro = paired.loc[paired['trajectory_group'] == 'immune_desertification','delta_neuro_PT_minus_DX'].values
rb_imm   = paired.loc[paired['trajectory_group'] == 'immune_rebuilding',     'delta_immune_PT_minus_DX'].values
ds_imm   = paired.loc[paired['trajectory_group'] == 'immune_desertification','delta_immune_PT_minus_DX'].values

u_neuro, p_neuro = mannwhitneyu(rb_neuro, ds_neuro, alternative='less')
u_imm,   p_imm   = mannwhitneyu(rb_imm,   ds_imm,   alternative='greater')
_, p_neuro_2s = mannwhitneyu(rb_neuro, ds_neuro, alternative='two-sided')
_, p_imm_2s   = mannwhitneyu(rb_imm,   ds_imm,   alternative='two-sided')

paired.reset_index(names='patient').to_csv(csvdir + 'fig2a_deltas_per_patient.csv', index=False)

stats_df = pd.DataFrame([
    {'metric': 'delta_neuro_PT_minus_DX',  'group1': 'immune_rebuilding', 'group2': 'immune_desertification',
     'alternative': 'less',    'U_stat': float(u_neuro), 'p_value': float(p_neuro),
     'p_value_two_sided': float(p_neuro_2s),
     'n_group1': int(len(rb_neuro)), 'n_group2': int(len(ds_neuro)),
     'min_possible_one_sided_p_with_n3_vs_n2': 0.1},
    {'metric': 'delta_immune_PT_minus_DX', 'group1': 'immune_rebuilding', 'group2': 'immune_desertification',
     'alternative': 'greater', 'U_stat': float(u_imm),   'p_value': float(p_imm),
     'p_value_two_sided': float(p_imm_2s),
     'n_group1': int(len(rb_imm)),   'n_group2': int(len(ds_imm)),
     'min_possible_one_sided_p_with_n3_vs_n2': 0.1},
])
stats_df.to_csv(csvdir + 'fig2a_mannwhitney_stats.csv', index=False)
display(stats_df)

## Suppl Table 2 source: Xenium gene panel

In [ ]:
adata = sc.read_h5ad(outdir_h5ad + 'xenium_integrated_labeled.h5ad')
pd.Series(adata.var_names, name='var_name').to_csv(
    csvdir + 'suppl_table_2_xenium_panel.csv', index=False
)

## Dotplot of main cell-type markers (Fig 1c)

In [ ]:
genes_to_plot = [
    'VWF', 'PTPRB', 'PDGFRA', 'DCN', 'S100B', 'CDH19',
    'PHOX2B', 'SRRM4', 'ITGAX', 'MS4A6A', 'IGHM', 'CD22', 'CD3E', 'CD96',
]
dp = sc.pl.dotplot(
    adata, genes_to_plot, groupby='celltype', cmap='YlOrRd',
    return_fig=True, show=False, standard_scale='var',
)
dp = dp.style(smallest_dot=0.01, largest_dot=70)
fig = dp.fig
fig.set_size_inches(7.5/2.54, 5/2.54, forward=True)
fig.tight_layout()
fig.savefig(figdir + 'celltype_dotplot.pdf', bbox_inches='tight')
plt.close(fig)

## T-cell subtype UMAP (Ext Fig 3a)

In [ ]:
adata_T = sc.read_h5ad(outdir_h5ad + 'T_labeled.h5ad')
adata_T = normalize_sample_meta(adata_T)

umap_df = pd.DataFrame({
    'id':        adata_T.obs_names,
    'umap1':     adata_T.obsm['X_umap'][:, 0],
    'umap2':     adata_T.obsm['X_umap'][:, 1],
    'T_subtype': adata_T.obs['T_subtype'].values,
})
umap_df.to_csv(csvdir + 'ext_fig3a_T_subtypes.csv', index=False)

In [ ]:
T_colors = {
    'Cytotoxic T':     '#ffffd9',
    'Naive/CM T':      '#c6e9b4',
    'Proliferating T': '#40b5c4',
    'Treg':            '#225da8',
    'γδT':             '#081d58', 'gdT': '#081d58',
}
umap_df = pd.read_csv(csvdir + 'ext_fig3a_T_subtypes.csv')
fig, ax = plt.subplots(figsize=(9/2.54, 7/2.54))
for st, sub in umap_df.groupby('T_subtype'):
    ax.scatter(sub['umap1'], sub['umap2'], s=2,
               c=T_colors.get(st, '#999999'),
               linewidths=0, alpha=0.8, label=st)
ax.legend(loc='right'); ax.set_title('T cell subtypes')
ax.set_xticks([]); ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)
fig.savefig(figdir + 'umap_T_subtype.png', dpi=600, bbox_inches='tight', transparent=True)
plt.show()

## T-cell subtype composition by timepoint (Ext Fig 3c)

T-cell subtype proportions aggregated across patients per timepoint
(`ext_fig3c_T_subtype_prop.csv`), per-patient per-timepoint proportions
(`suppl_table_3_ind_T_cell_subtype_prop.csv`) and paired Wilcoxon
DX vs PT tests per subtype (`ext_fig3c_stats.csv`).

In [ ]:
# Aggregate timepoint -> T-cell subtype composition.
df = (
    adata_T.obs
    .groupby(['timepoint', 'T_subtype']).size().reset_index(name='count')
)
df['prop'] = df.groupby('timepoint')['count'].transform(lambda x: x / x.sum())
df.to_csv(csvdir + 'ext_fig3c_T_subtype_prop.csv', index=False)

# Per-sample per-timepoint T-cell subtype proportions.
tbl = (
    adata_T.obs
    .groupby(['sample', 'timepoint', 'T_subtype']).size().reset_index(name='count')
)
tbl['prop'] = tbl.groupby(['sample', 'timepoint'])['count'].transform(lambda x: x / x.sum())
prop_wide = (
    tbl.pivot_table(index=['sample', 'timepoint'], columns='T_subtype',
                    values='prop', fill_value=0).reset_index()
)
prop_wide.to_csv(csvdir + 'suppl_table_3_ind_T_cell_subtype_prop.csv', index=False)

In [ ]:
# Dotplot of T-cell subtype proportions per timepoint.
df = pd.read_csv(csvdir + 'ext_fig3c_T_subtype_prop.csv')
df['timepoint'] = pd.Categorical(df['timepoint'], categories=['PT', 'DX'], ordered=True)

fig, ax = plt.subplots(figsize=(9/2.54, 4.6/2.54))
sns.scatterplot(data=df, x='T_subtype', y='timepoint', size='prop',
                sizes=(5, 90), hue='count', palette='RdBu', ax=ax)
ypos = {cat: i for i, cat in enumerate(df['timepoint'].cat.categories)}
for _, row in df.iterrows():
    ax.text(row['T_subtype'], ypos[row['timepoint']] + 0.2,
            f"{int(row['count'])}\n({row['prop']:.2f})", ha='center', va='bottom', fontsize=5)
plt.setp(ax.get_xticklabels(), rotation=90, ha='right')
ax.set_ylim(-0.2, 1.8)
ax.legend(bbox_to_anchor=(1, 1), loc='upper left', frameon=False)
fig.tight_layout()
fig.savefig(figdir + 'dotplot_count_timepoint_T.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Paired Wilcoxon DX vs PT per T-cell subtype.
prop_wide = pd.read_csv(csvdir + 'suppl_table_3_ind_T_cell_subtype_prop.csv')
subtypes = [c for c in prop_wide.columns if c not in ('sample', 'timepoint')]

tmp = prop_wide.copy()
tmp['patient'] = tmp['sample'].str.replace(r'_(DX|PT)$', '', regex=True)
long = tmp.melt(id_vars=['patient', 'timepoint'], value_vars=subtypes,
                var_name='T_subtype', value_name='prop')

rows = []
for st in subtypes:
    d = long[long['T_subtype'] == st]
    wide = d.pivot(index='patient', columns='timepoint', values='prop').dropna(subset=['DX', 'PT'])
    dx = wide['DX'].values; pt = wide['PT'].values; delta = pt - dx
    try:
        stat, p = wilcoxon(pt, dx, alternative='two-sided', zero_method='wilcox', method='exact')
    except Exception:
        stat, p = wilcoxon(pt, dx, alternative='two-sided', zero_method='wilcox')
    rows.append({
        'T_subtype': st, 'n_pairs': len(wide),
        'mean_DX': float(np.mean(dx)),  'mean_PT': float(np.mean(pt)),
        'mean_delta_PT_minus_DX': float(np.mean(delta)),
        'median_DX': float(np.median(dx)), 'median_PT': float(np.median(pt)),
        'median_delta_PT_minus_DX': float(np.median(delta)),
        'wilcoxon_stat': float(stat), 'p_value': float(p),
    })
stats_df = pd.DataFrame(rows)
_, qvals, _, _ = multipletests(stats_df['p_value'].values, alpha=0.05, method='fdr_bh')
stats_df['p_value_fdr'] = qvals
stats_df.to_csv(csvdir + 'ext_fig3c_stats.csv', index=False)
display(stats_df)

## Xenium Explorer celltype color export (per-sample)


In [ ]:
sample_id = 'Patient5_DX'
sub = adata[adata.obs['sample'] == sample_id].copy()

if 'celltype_colors' in adata.uns and pd.api.types.is_categorical_dtype(adata.obs['celltype']):
    cats = adata.obs['celltype'].cat.categories
    colors = adata.uns['celltype_colors']
    celltype_color_map = {c: colors[i] for i, c in enumerate(cats)}
else:
    celltype_color_map = {c: '#E6E6E6' for c in pd.unique(adata.obs['celltype'])}

df_out = pd.DataFrame({
    'cell_id': sub.obs['cell_id'].values,
    'group':   sub.obs['celltypes_all'].values,
    'color':   sub.obs['celltype'].map(celltype_color_map).values,
})
df_out.to_csv(csvdir + f'{sample_id}_celltypes_colors.csv', index=False)